# Stage 2 of 3 — Rule Building

**This notebook does ONE thing**: canonicalize `mined_pairs.jsonl` (from Stage 1)
into a deduped, frequency-ranked `rules.json`. No mining, no optimization —
this is the ~40-minute stage, and the only thing it does is that.

**Setup:**
1. **Add Input → Notebook Output → (your username) → `01-mining`** (the saved
   version of Stage 1). This is what makes `mined_pairs.jsonl` show up under
   `/kaggle/input/` here — a real file handoff between two notebooks, not a
   flag inside one shared file.
2. **Internet: On** (needed for `pip install`).

**When this finishes:** **Save Version**. Its Output (`rules.json`) becomes
Stage 3's (`03_optimize.ipynb`) input, the same way.

In [ ]:
import glob
print(glob.glob('/kaggle/input/*'))
print(glob.glob('/kaggle/input/**/*', recursive=True)[:20])

In [ ]:
# Base package only -- rule-building needs qiskit (imported by qcs_pipeline's
# gate_sequence module) but not the [mining] extra's pandas/pyarrow/pyzx,
# since mined_pairs.jsonl is plain JSON, not parquet.
!pip install -q "quantum-circuit-smell-intelligence @ git+https://github.com/veerakrish/quantum-circuit-smell-intelligence.git"

In [ ]:
try:
    import qcs_pipeline
    print("qcs_pipeline imported OK from:", qcs_pipeline.__file__)
except ModuleNotFoundError as e:
    raise RuntimeError(
        "qcs_pipeline is not importable. Check the pip install cell's output "
        "above, or restart the session if you just picked up a code update."
    ) from e

In [ ]:
# Locate mined_pairs.jsonl -- search broadly rather than assuming a fixed
# path, same reasoning as Stage 1's dataset discovery: the exact mount point
# for an attached notebook's output isn't worth hardcoding.
import glob
from pathlib import Path

matches = glob.glob("/kaggle/input/**/mined_pairs.jsonl", recursive=True)
if not matches:
    raise FileNotFoundError(
        "mined_pairs.jsonl not found under /kaggle/input. Did you attach "
        "Stage 1's saved notebook output as an input to this notebook? "
        "(Add Input -> Notebook Output -> your username -> 01-mining)"
    )
MINED_PAIRS_PATH = Path(matches[0])
print(f"Using {MINED_PAIRS_PATH} ({MINED_PAIRS_PATH.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# The only real work this notebook does.
#
# The raw rule count overstates what's actually usable: many mined rewrites
# (rz->rz is the biggest offender) have angle changes that aren't a simple
# copy/negate/sum of the removed gates' angles -- Qiskit resynthesized a
# whole gate sequence into a new closed-form angle rather than a linear
# combination of the old one(s). Those are kept for audit purposes but
# flagged "structural_only" and never auto-applied in Stage 3.
from qcs_pipeline.rules.rule_database import build_rule_database
from qcs_pipeline.detector.smell_detector import summarize_applicability, is_structural_only

db = build_rule_database(MINED_PAIRS_PATH, min_frequency=2)
db.to_json(Path("/kaggle/working/rules.json"))

summary = summarize_applicability(db)
print(f"{summary['total']} unique rules mined:")
print(f"  {summary['applicable']} applicable (safe to auto-apply)")
print(f"  {summary['conflicting']} conflicting (same pattern, multiple observed rewrites)")
print(f"  {summary['structural_only']} structural-only (rewrite exists, angle not symbolically resolved)")

entries = db.entries()
print("\nTop 10 by frequency:")
for e in sorted(entries, key=lambda x: -x.frequency)[:10]:
    flags = []
    if e.conflict:
        flags.append("CONFLICT")
    if is_structural_only(e):
        flags.append("STRUCTURAL-ONLY")
    flag_str = f" [{', '.join(flags)}]" if flags else ""
    print(e.frequency, [op.name for op in e.pattern], "->", [op.name for op in e.rewrite], flag_str)

## Done

`rules.json` is in `/kaggle/working/`. **Save Version** now — that's what Stage 3
(`03_optimize.ipynb`) attaches as its input.